<a href="https://colab.research.google.com/github/Frans1828/scikit-learn-cookbook/blob/main/06.%20Advanced%20Logistic%20Regression/ch06_logistic_regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

---
## Identitas Mahasiswa

| Informasi | Keterangan |
|---|---|
| **Nama** | Fransisco Sitepu |
| **NIM** | 101032300004 |
| **Kelas** | TK 47 03 |
| **Mata Kuliah** | Machine Learning |
| **Institusi** | Telkom University |

---


# Chapter 6: Advanced Logistic Regression

## Overview

Logistic Regression adalah algoritma **klasifikasi linear** yang memodelkan probabilitas kelas menggunakan fungsi **sigmoid (logit)**. Meskipun namanya "regression", ini adalah algoritma klasifikasi.

### Fungsi Sigmoid

$$P(y=1 | \mathbf{x}) = \sigma(\mathbf{w}^\top \mathbf{x} + b) = \frac{1}{1 + e^{-(\mathbf{w}^\top \mathbf{x} + b)}}$$

Fungsi sigmoid memetakan sembarang nilai real ke $(0, 1)$, yang dapat diinterpretasikan sebagai probabilitas.

### Log-Odds (Logit)

$$\log \frac{P(y=1|\mathbf{x})}{P(y=0|\mathbf{x})} = \mathbf{w}^\top \mathbf{x} + b$$

### Binary Cross-Entropy Loss (Log-Loss)

$$\mathcal{L} = -\frac{1}{n}\sum_{i=1}^n \left[y_i \log(\hat{p}_i) + (1-y_i) \log(1-\hat{p}_i)\right]$$


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer, load_iris, make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, classification_report,
    roc_auc_score, roc_curve, precision_recall_curve,
    log_loss, confusion_matrix, ConfusionMatrixDisplay
)

# Load datasets
cancer = load_breast_cancer()
X_cancer, y_cancer = cancer.data, cancer.target

iris = load_iris()
X_iris, y_iris = iris.data, iris.target

print(f'Breast Cancer: X={X_cancer.shape}, classes={cancer.target_names}')
print(f'Iris: X={X_iris.shape}, classes={iris.target_names}')


## 6.1 Visualisasi Fungsi Sigmoid

Memahami bagaimana sigmoid bekerja dan pengaruh weight $w$ terhadap "kecuraman" kurva:


In [ ]:
# Visualize sigmoid function
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

z = np.linspace(-8, 8, 300)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Basic sigmoid
axes[0].plot(z, sigmoid(z), 'b-', lw=2.5)
axes[0].axhline(0.5, color='red', linestyle='--', alpha=0.6, label='Decision boundary (p=0.5)')
axes[0].axvline(0, color='gray', linestyle='--', alpha=0.6)
axes[0].fill_between(z, sigmoid(z), 0.5, where=(z > 0), alpha=0.1, color='green', label='P(y=1) > 0.5')
axes[0].fill_between(z, sigmoid(z), 0.5, where=(z < 0), alpha=0.1, color='red', label='P(y=1) < 0.5')
axes[0].set_xlabel('z = w^T x + b')
axes[0].set_ylabel('sigma(z) = P(y=1|x)')
axes[0].set_title('Sigmoid Function')
axes[0].legend(fontsize=8)
axes[0].set_ylim(-0.05, 1.05)

# Effect of weight scale
for w in [0.5, 1.0, 2.0, 4.0]:
    axes[1].plot(z, sigmoid(w * z), lw=2, label=f'w={w}')
axes[1].axhline(0.5, color='gray', linestyle='--', alpha=0.5)
axes[1].set_xlabel('x')
axes[1].set_ylabel('P(y=1|x)')
axes[1].set_title('Effect of Weight Scale on Sigmoid')
axes[1].legend()

plt.tight_layout()
plt.show()


## 6.2 Binary Logistic Regression

Logistic Regression dioptimasi menggunakan **Maximum Likelihood Estimation (MLE)**. Regularisasi dapat ditambahkan untuk mencegah overfitting:

- **L2 regularization (default)**: $\min \mathcal{L} + \frac{1}{2C}\|\mathbf{w}\|^2$
- **L1 regularization**: $\min \mathcal{L} + \frac{1}{C}\|\mathbf{w}\|_1$ (sparse solutions)
- **Elasticnet**: Kombinasi L1 dan L2

Hyperparameter $C$ adalah **inverse regularization strength**: $C$ kecil = regularisasi kuat.


In [ ]:
# Binary classification on breast cancer
X_tr, X_te, y_tr, y_te = train_test_split(
    X_cancer, y_cancer, test_size=0.25, random_state=42, stratify=y_cancer
)

# Test different C values
C_values = [0.001, 0.01, 0.1, 1.0, 10.0, 100.0]
train_accs, test_accs, cv_accs, n_nonzero = [], [], [], []

for C in C_values:
    pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('lr', LogisticRegression(C=C, max_iter=5000, random_state=42))
    ])
    pipe.fit(X_tr, y_tr)
    train_accs.append(pipe.score(X_tr, y_tr))
    test_accs.append(pipe.score(X_te, y_te))
    cv_accs.append(cross_val_score(pipe, X_cancer, y_cancer, cv=5).mean())
    n_nonzero.append(np.sum(np.abs(pipe['lr'].coef_[0]) > 1e-4))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

log_C = np.log10(C_values)
axes[0].plot(log_C, train_accs, 'b-o', label='Train Accuracy')
axes[0].plot(log_C, test_accs, 'r-s', label='Test Accuracy')
axes[0].plot(log_C, cv_accs, 'g-^', label='5-Fold CV Accuracy')
axes[0].set_xlabel('log10(C)')
axes[0].set_ylabel('Accuracy')
axes[0].set_title('Logistic Regression: Effect of C (L2)')
axes[0].legend()

axes[1].plot(log_C, n_nonzero, 'purple', marker='D', lw=2)
axes[1].set_xlabel('log10(C)')
axes[1].set_ylabel('Number of Non-Zero Coefficients')
axes[1].set_title('Regularization Effect on Model Sparsity')

plt.tight_layout()
plt.show()

best_C = C_values[np.argmax(cv_accs)]
print(f'Best C by CV: {best_C}, CV Accuracy: {max(cv_accs):.4f}')


## 6.3 ROC Curve dan AUC

ROC (Receiver Operating Characteristic) curve menggambarkan tradeoff antara **True Positive Rate (Recall/Sensitivity)** dan **False Positive Rate** pada berbagai threshold:

$$TPR = \frac{TP}{TP + FN}, \quad FPR = \frac{FP}{FP + TN}$$

**AUC (Area Under Curve)** menginterpretasikan kemampuan discriminatif model secara keseluruhan:
- AUC = 1.0: model sempurna
- AUC = 0.5: model random (tidak lebih baik dari menebak)


In [ ]:
from sklearn.model_selection import StratifiedKFold

# Best model
best_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('lr', LogisticRegression(C=best_C, max_iter=5000, random_state=42))
])
best_pipe.fit(X_tr, y_tr)
y_proba = best_pipe.predict_proba(X_te)[:, 1]

# ROC Curve
fpr, tpr, thresholds = roc_curve(y_te, y_proba)
auc_score = roc_auc_score(y_te, y_proba)

# Precision-Recall Curve
prec, rec, _ = precision_recall_curve(y_te, y_proba)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ROC
axes[0].plot(fpr, tpr, 'b-', lw=2, label=f'LR (AUC={auc_score:.4f})')
axes[0].plot([0, 1], [0, 1], 'k--', lw=1, label='Random (AUC=0.5)')
axes[0].fill_between(fpr, tpr, alpha=0.1, color='blue')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate (Recall)')
axes[0].set_title(f'ROC Curve - Breast Cancer\n(C={best_C})')
axes[0].legend()
axes[0].set_xlim(0, 1); axes[0].set_ylim(0, 1.02)

# Precision-Recall
axes[1].plot(rec, prec, 'r-', lw=2)
axes[1].fill_between(rec, prec, alpha=0.1, color='red')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curve')

plt.tight_layout()
plt.show()

print(f'AUC-ROC: {auc_score:.4f}')
print(f'Log-Loss: {log_loss(y_te, y_proba):.4f}')
print(classification_report(y_te, best_pipe.predict(X_te), target_names=cancer.target_names))


## 6.4 Multiclass Logistic Regression

Untuk klasifikasi multiclass, scikit-learn mendukung dua strategi:

**One-vs-Rest (OvR)**: melatih $K$ classifier biner, satu per kelas:
$$P(y=k|\mathbf{x}) = \sigma(\mathbf{w}_k^\top \mathbf{x})$$

**Multinomial (Softmax)**: satu model tunggal untuk semua kelas:
$$P(y=k|\mathbf{x}) = \frac{e^{\mathbf{w}_k^\top \mathbf{x}}}{\sum_{j=1}^K e^{\mathbf{w}_j^\top \mathbf{x}}}$$


In [ ]:
# Multiclass on iris
X_tr_i, X_te_i, y_tr_i, y_te_i = train_test_split(
    X_iris, y_iris, test_size=0.3, random_state=42, stratify=y_iris
)

configs = [
    ('OvR, C=1', LogisticRegression(multi_class='ovr', C=1.0, max_iter=1000, random_state=42)),
    ('Multinomial, C=1', LogisticRegression(multi_class='multinomial', C=1.0, max_iter=1000, random_state=42, solver='lbfgs')),
    ('OvR, C=10', LogisticRegression(multi_class='ovr', C=10.0, max_iter=1000, random_state=42)),
]

print('=== Multiclass Logistic Regression (Iris) ===')
for name, lr in configs:
    pipe = Pipeline([('scaler', StandardScaler()), ('lr', lr)])
    cv_acc = cross_val_score(pipe, X_iris, y_iris, cv=5).mean()
    pipe.fit(X_tr_i, y_tr_i)
    test_acc = pipe.score(X_te_i, y_te_i)
    print(f'{name:<25}: test={test_acc:.4f}, CV={cv_acc:.4f}')

# Best model: Multinomial
best_multi = Pipeline([
    ('scaler', StandardScaler()),
    ('lr', LogisticRegression(multi_class='multinomial', C=10.0, max_iter=1000, random_state=42, solver='lbfgs'))
])
best_multi.fit(X_tr_i, y_tr_i)
y_pred_i = best_multi.predict(X_te_i)

print(f'\nBest Multinomial Classification Report:')
print(classification_report(y_te_i, y_pred_i, target_names=iris.target_names))

# Confusion Matrix
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_estimator(best_multi, X_te_i, y_te_i,
    display_labels=iris.target_names, cmap='Blues', ax=ax)
ax.set_title('Multinomial LR - Iris Confusion Matrix')
plt.tight_layout()
plt.show()


## Ringkasan: Key Takeaways Chapter 6

| Konsep | Poin Kunci |
|---|---|
| **Sigmoid** | Memetakan nilai real ke (0,1); outputnya adalah probabilitas kelas |
| **Log-Loss** | Fungsi loss yang dipilih untuk training; lebih sensitif terhadap prediksi yang sangat yakin namun salah |
| **Regularisasi C** | C kecil = regularisasi kuat (bias tinggi); C besar = regularisasi lemah (variance tinggi) |
| **L1 vs L2** | L1 → sparse coefficients (feature selection); L2 → shrink all coefficients |
| **OvR vs Multinomial** | Multinomial lebih baik untuk kelas yang saling berhubungan; OvR lebih cepat |
| **AUC-ROC** | Mengukur kemampuan discriminatif; gunakan untuk evaluasi pada data tidak seimbang |

### Solvers Logistic Regression di scikit-learn
| Solver | Binary | Multiclass | L1 | L2 | Elasticnet |
|---|---|---|---|---|---|
| `lbfgs` | Yes | multinomial | No | Yes | No |
| `liblinear` | Yes | OvR | Yes | Yes | No |
| `saga` | Yes | multinomial | Yes | Yes | Yes |
